# Clustering with KMeans

## What you'll learn
- Unsupervised learning: finding groups when there is no answer column
- Why scaling matters for distance-based methods
- KMeans, and how to choose the number of clusters (elbow and silhouette)
- How to describe each cluster so the business can use it

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values
print(df.shape)

## The goal

The sales team wants to group accounts by **buying behaviour** so each group gets the
right service model. There is no correct answer column, so this is **unsupervised**
learning: the algorithm finds structure on its own.

**KMeans** places `k` centre points, assigns each account to the nearest centre, moves each
centre to the middle of its accounts, and repeats until nothing changes.

In [ ]:
BEHAVIOUR = ["orders_last_year", "avg_order_value", "discount_pct", "support_tickets"]
X = df[BEHAVIOUR]
X.describe().round(1)

## Scale first

KMeans measures distance. Order value is in thousands while tickets are single digits,
so without scaling, order value would decide everything. `StandardScaler` puts every column
on the same footing.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

## How many clusters?

Two common checks:

- **Elbow:** inertia (how tightly accounts sit around their centres) always falls as `k`
  grows; look for the point where it stops falling fast.
- **Silhouette score:** from -1 to 1, how much closer each account is to its own cluster
  than to the next one. Higher is better.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(2, 9)
inertia, silhouette = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(ks), inertia, marker="o")
axes[0].set_title("Elbow: inertia by k")
axes[1].plot(list(ks), silhouette, marker="o")
axes[1].set_title("Silhouette score by k")
for ax in axes:
    ax.set_xlabel("number of clusters (k)")
plt.show()

The chart is a guide, not a rule. A number of clusters the business can act on matters as
much as the score. We will use 4.

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_scaled)
df["cluster"] = kmeans.labels_
df["cluster"].value_counts().sort_index()

## Describe the clusters

A cluster number means nothing to a sales team. Profile each cluster in the original units
and give it a name.

In [ ]:
profile = df.groupby("cluster")[BEHAVIOUR + ["spend_last_year"]].mean().round(1)
profile["accounts"] = df["cluster"].value_counts().sort_index()
profile

In [ ]:
# How the clusters line up with the known segments (the model never saw this column).
pd.crosstab(df["cluster"], df["segment"])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(df["orders_last_year"], df["avg_order_value"], c=df["cluster"],
                     cmap="viridis", alpha=0.5)
ax.set_xlabel("Orders last year")
ax.set_ylabel("Average order value (USD)")
ax.set_title("Accounts coloured by cluster")
ax.legend(*scatter.legend_elements(), title="cluster")
plt.show()

The clusters largely rediscover the segments from buying behaviour alone, which is a good
sign. Accounts sitting in an "unexpected" cluster for their segment are worth a look: a
clinic that buys like a distributor may need a different service model.

## Practice

1. Re-run KMeans with 3 clusters and profile them. Are they easier or harder to name?
2. Run KMeans on the **unscaled** data. How do the clusters change, and why?
3. Add `avg_days_to_pay` to the behaviour columns (fill its missing values first, or use
   the tidied `df`). What new pattern appears?
4. Write one-line names for each of the 4 clusters based on the profile table.

In [ ]:
# Your practice code here

## Summary

- Clustering finds groups without an answer column.
- Scale features first: KMeans is based on distance.
- Use the elbow chart and silhouette score as guides, and pick a `k` the business can use.
- Profile clusters in original units and name them; compare with known labels when you have them.

## Practice Solutions

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

BEHAVIOUR = ["orders_last_year", "avg_order_value", "discount_pct", "support_tickets"]
X_scaled = StandardScaler().fit_transform(df[BEHAVIOUR])

# 1.
df["k3"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_scaled)
print(df.groupby("k3")[BEHAVIOUR].mean().round(1), "\n")

# 2. Without scaling, avg_order_value (the largest numbers) dominates the distances,
#    so clusters become little more than order-value bands.
df["raw"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(df[BEHAVIOUR])
print(df.groupby("raw")[BEHAVIOUR].mean().round(1), "\n")

# 3.
cols = BEHAVIOUR + ["avg_days_to_pay"]
df["k4_pay"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(
    StandardScaler().fit_transform(df[cols]))
print(df.groupby("k4_pay")[cols].mean().round(1))

# 4. Names depend on your profile table, for example:
#    "distributors: most orders, biggest discounts", "hospitals: large regular orders",
#    "clinics and laboratories: small orders", "accounts with many support tickets".